# Cattle Muzzle Detection - Training on Kaggle GPU

### Setup Instructions:
1. Upload your dataset as a Kaggle Dataset (see Step 2 below)
2. Enable GPU: **Settings (right panel) > Accelerator > GPU T4 x2 or P100**
3. Run all cells in order
4. Download trained model files from the Output section

---

## Step 1: Check GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU! Go to Settings > Accelerator > GPU")

import os
import glob

# List all input datasets
print("Available input datasets:")
for root, dirs, files in os.walk('/kaggle/input/'):
    level = root.replace('/kaggle/input/', '').count(os.sep)
    if level < 4:
        print(f"{'  ' * level}{root}/")

# Dataset path
DATASET_PATH = '/kaggle/input/datasets/darvesh748/cattle-muzzle-db/Cattle Muzzle - DB/Original'

# Auto-find fallback
if not os.path.exists(DATASET_PATH):
    possible = glob.glob('/kaggle/input/**/Original', recursive=True)
    if possible:
        DATASET_PATH = possible[0]
        print(f"Auto-detected: {DATASET_PATH}")
    else:
        print("ERROR: Could not find 'Original' folder. Check your dataset upload.")

print(f"\nDataset path: {DATASET_PATH}")
if os.path.exists(DATASET_PATH):
    folders = sorted([d for d in os.listdir(DATASET_PATH) 
                      if os.path.isdir(os.path.join(DATASET_PATH, d)) and d.isdigit()], key=int)
    print(f"Cattle folders: {folders}")
    print(f"Total cattle: {len(folders)}")
else:
    print("PATH NOT FOUND!")

In [ ]:
import os
import glob

# List all input datasets
print("Available input datasets:")
for item in os.listdir('/kaggle/input/'):
    print(f"  /kaggle/input/{item}")

# Auto-find the Original folder
possible = glob.glob('/kaggle/input/**/Original', recursive=True)
if possible:
    DATASET_PATH = possible[0]
    print(f"\nFound dataset at: {DATASET_PATH}")
else:
    # Manual fallback - UPDATE THIS if auto-detect fails
    DATASET_PATH = '/kaggle/input/cattle-muzzle-db/Cattle Muzzle - DB/Original'
    print(f"\nUsing manual path: {DATASET_PATH}")
    if not os.path.exists(DATASET_PATH):
        print("PATH NOT FOUND! Listing input contents to help you find it:")
        for root, dirs, files in os.walk('/kaggle/input/'):
            level = root.replace('/kaggle/input/', '').count(os.sep)
            if level < 3:
                indent = ' ' * 2 * level
                print(f"{indent}{os.path.basename(root)}/")

# Verify
if os.path.exists(DATASET_PATH):
    folders = sorted([d for d in os.listdir(DATASET_PATH) 
                      if os.path.isdir(os.path.join(DATASET_PATH, d)) and d.isdigit()], key=int)
    print(f"Cattle folders: {folders}")
    print(f"Total cattle: {len(folders)}")

## Step 3: Define Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import ResNet50_Weights


class EmbeddingNetwork(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        backbone = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embedding_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        x = F.normalize(x, p=2, dim=1)
        return x


class SiameseNetwork(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.embedding_net = EmbeddingNetwork(embedding_dim)

    def forward(self, img1, img2):
        emb1 = self.embedding_net(img1)
        emb2 = self.embedding_net(img2)
        return emb1, emb2

    def get_embedding(self, img):
        return self.embedding_net(img)


class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        distance = F.pairwise_distance(emb1, emb2)
        loss = label * distance.pow(2) + (1 - label) * F.relu(self.margin - distance).pow(2)
        return loss.mean()


print("Model defined!")

## Step 4: Define Dataset & Transforms

In [ ]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Config
EMBEDDING_DIM = 128
IMAGE_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 30
FREEZE_EPOCHS = 5
MARGIN = 1.0
PAIRS_PER_EPOCH = 5000

# Kaggle output path
OUTPUT_DIR = '/kaggle/working'


def get_transforms(is_training=True):
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    )
    if is_training:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomCrop(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(IMAGE_SIZE),
            transforms.ToTensor(),
            normalize,
        ])


def split_cattle_ids(dataset_path):
    all_ids = sorted(
        [d for d in os.listdir(dataset_path)
         if os.path.isdir(os.path.join(dataset_path, d)) and d.isdigit()],
        key=int,
    )
    random.seed(42)
    random.shuffle(all_ids)
    n = len(all_ids)
    train_end = int(n * 0.7)
    val_end = train_end + int(n * 0.15)
    return all_ids[:train_end], all_ids[train_end:val_end], all_ids[val_end:]


class CattleMuzzleDataset(Dataset):
    def __init__(self, dataset_path, cattle_ids, transform=None, pairs_per_epoch=5000):
        self.dataset_path = dataset_path
        self.cattle_ids = cattle_ids
        self.transform = transform or get_transforms(is_training=True)
        self.pairs_per_epoch = pairs_per_epoch
        self.images = {}
        total = 0
        for cid in self.cattle_ids:
            folder = os.path.join(dataset_path, cid)
            imgs = [os.path.join(folder, f) for f in os.listdir(folder)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if len(imgs) >= 2:
                self.images[cid] = imgs
                total += len(imgs)
        self.cattle_ids = list(self.images.keys())
        print(f"Loaded {total} images from {len(self.cattle_ids)} cattle")

    def __len__(self):
        return self.pairs_per_epoch

    def __getitem__(self, idx):
        if random.random() < 0.5:
            cid = random.choice(self.cattle_ids)
            img1_path, img2_path = random.sample(self.images[cid], 2)
            label = 1.0
        else:
            cid1, cid2 = random.sample(self.cattle_ids, 2)
            img1_path = random.choice(self.images[cid1])
            img2_path = random.choice(self.images[cid2])
            label = 0.0
        img1 = self.transform(Image.open(img1_path).convert('RGB'))
        img2 = self.transform(Image.open(img2_path).convert('RGB'))
        return img1, img2, torch.tensor(label, dtype=torch.float32)


class SingleImageDataset(Dataset):
    def __init__(self, dataset_path, cattle_ids, transform=None):
        self.transform = transform or get_transforms(is_training=False)
        self.samples = []
        for cid in cattle_ids:
            folder = os.path.join(dataset_path, cid)
            for f in os.listdir(folder):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(folder, f), cid))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, cid = self.samples[idx]
        img = self.transform(Image.open(path).convert('RGB'))
        return img, cid, path


print("Dataset classes defined!")

## Step 5: Train the Model

In [ ]:
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# Split data
train_ids, val_ids, test_ids = split_cattle_ids(DATASET_PATH)
print(f"Split: {len(train_ids)} train, {len(val_ids)} val, {len(test_ids)} test")

# Datasets
train_dataset = CattleMuzzleDataset(DATASET_PATH, train_ids,
                                     transform=get_transforms(True), pairs_per_epoch=PAIRS_PER_EPOCH)
val_dataset = CattleMuzzleDataset(DATASET_PATH, val_ids,
                                   transform=get_transforms(False), pairs_per_epoch=500)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Model
model = SiameseNetwork(embedding_dim=EMBEDDING_DIM).to(device)
criterion = ContrastiveLoss(margin=MARGIN)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Freeze backbone
for param in model.embedding_net.features.parameters():
    param.requires_grad = False
print(f"Backbone frozen for first {FREEZE_EPOCHS} epochs")

MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, 'siamese_resnet50.pth')
REF_SAVE_PATH = os.path.join(OUTPUT_DIR, 'reference_embeddings.npy')

best_val_loss = float('inf')
best_epoch = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # Unfreeze backbone
    if epoch == FREEZE_EPOCHS + 1:
        for param in model.embedding_net.features.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam([
            {'params': model.embedding_net.features.parameters(), 'lr': LEARNING_RATE * 0.1},
            {'params': model.embedding_net.head.parameters(), 'lr': LEARNING_RATE},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
        print(">>> Backbone UNFROZEN - fine-tuning")

    # Train
    model.train()
    running_loss = 0.0
    start = time.time()
    for img1, img2, label in train_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        optimizer.zero_grad()
        emb1, emb2 = model(img1, img2)
        loss = criterion(emb1, emb2, label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    avg_train = running_loss / len(train_loader)

    # Validate
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for img1, img2, label in val_loader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            emb1, emb2 = model(img1, img2)
            loss = criterion(emb1, emb2, label)
            val_loss += loss.item()
            dist = F.pairwise_distance(emb1, emb2)
            pred = (dist < MARGIN / 2).float()
            correct += (pred == label).sum().item()
            total += label.size(0)

    avg_val = val_loss / len(val_loader)
    val_acc = correct / total if total > 0 else 0
    elapsed = time.time() - start

    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    history['val_acc'].append(val_acc)

    saved = ''
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': avg_val,
            'val_acc': val_acc,
        }, MODEL_SAVE_PATH)
        saved = ' [SAVED]'

    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
          f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
          f"Acc: {val_acc:.1%} | Time: {elapsed:.0f}s{saved}")

print(f"\nDone! Best: epoch {best_epoch}, val_loss={best_val_loss:.4f}")

## Step 6: Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(history['val_acc'], label='Val Accuracy', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Step 7: Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']}")

# Test
test_dataset = CattleMuzzleDataset(DATASET_PATH, test_ids,
                                    transform=get_transforms(False), pairs_per_epoch=2000)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

correct = 0
total = 0
all_distances = []
all_labels = []

with torch.no_grad():
    for img1, img2, label in test_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        emb1, emb2 = model(img1, img2)
        dist = F.pairwise_distance(emb1, emb2)
        pred = (dist < MARGIN / 2).float()
        correct += (pred == label).sum().item()
        total += label.size(0)
        all_distances.extend(dist.cpu().numpy())
        all_labels.extend(label.cpu().numpy())

print(f"Test Accuracy: {correct/total:.1%} ({correct}/{total})")

# Distance distribution plot
all_distances = np.array(all_distances)
all_labels = np.array(all_labels)

plt.figure(figsize=(10, 5))
plt.hist(all_distances[all_labels == 1], bins=50, alpha=0.7, label='Same Cattle', color='green')
plt.hist(all_distances[all_labels == 0], bins=50, alpha=0.7, label='Different Cattle', color='red')
plt.axvline(x=MARGIN/2, color='black', linestyle='--', label=f'Threshold ({MARGIN/2})')
plt.xlabel('Euclidean Distance')
plt.ylabel('Count')
plt.title('Distance Distribution: Same vs Different Cattle')
plt.legend()
plt.grid(True)
plt.show()

## Step 8: Save Reference Embeddings

In [ ]:
# Compute reference embeddings for muzzle detection
ref_dataset = SingleImageDataset(DATASET_PATH, train_ids, transform=get_transforms(False))
ref_loader = DataLoader(ref_dataset, batch_size=32, shuffle=False, num_workers=2)

all_embs = []
with torch.no_grad():
    for imgs, _, _ in ref_loader:
        imgs = imgs.to(device)
        embs = model.get_embedding(imgs)
        all_embs.append(embs.cpu().numpy())

all_embs = np.concatenate(all_embs, axis=0)
mean_emb = np.mean(all_embs, axis=0)
np.save(REF_SAVE_PATH, mean_emb)
print(f"Reference embedding saved ({all_embs.shape[0]} images)")

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"\nFiles saved to /kaggle/working/:")
print(f"  1. siamese_resnet50.pth ({os.path.getsize(MODEL_SAVE_PATH)/1e6:.1f} MB)")
print(f"  2. reference_embeddings.npy")
print(f"\nDownload from the 'Output' tab (right panel)")
print(f"\nThen place them in:")
print(f"  C:\\Users\\darve\\AI_project\\cattle_muzzle_detection\\saved_models\\")
print(f"\nThen run: streamlit run app.py")